<a href="https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_9B_Generate_Reasoning_Examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate checked reasoning examples with Qwen3.5-9B

Step 1 of 2. This notebook makes thinking-mode training examples; [step 2](https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_2B_Thinking_Finetune.ipynb) fine-tunes
Qwen3.5-2B on them.

1. **Write tasks.** Qwen3.5-9B turns docs passages that contain GDScript into realistic coding requests, and API questions are taken from the [godot-docs-dataset](https://huggingface.co/datasets/arunsrajan/godot-docs-dataset). Some coding tasks are held out to evaluate the 2B later.
2. **Solve them in thinking mode** with the same tools as the [coding assistant notebook](https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_Godot_Coding_Assistant.ipynb): `search_godot_docs` and `check_gdscript` (Godot 4.7.2's own parser).
3. **Keep only checked examples.** A conversation becomes a training example only if the model finished thinking, searched the docs, checked its code (for coding tasks), and the GDScript in its final answer passes Godot's checker.
4. **Push them to a private Hugging Face dataset**, with the reasoning kept in every assistant turn.

**Runtime:** this needs more than a free T4. Use `Runtime > Change runtime type > A100 GPU`. On an L4, uncomment
`quantization = "fp8"` in section 4. vLLM generates all conversations in a batch together, which is much faster than one at a time.

**How long it takes** depends on how much the model thinks. Each batch prints its time, so after the first batch you
can lower `NUM_CODING_TASKS` or stop early: everything done so far is saved, and running the notebook again resumes.

**License:** Qwen3.5 is Apache-2.0, so you can train on its outputs. The docs are CC BY 3.0.

## 1. Install vLLM

vLLM needs newer torch and transformers versions than Unsloth, which is why generation and training are separate notebooks.

In [ ]:
!pip install -q vllm==0.30.0 datasets

## 2. Settings

To keep progress if Colab disconnects, mount Google Drive (`from google.colab import drive; drive.mount("/content/drive")`)
and set `OUTPUT_DIR` to a folder in it, such as `"/content/drive/MyDrive/reasoning_examples"`.

In [ ]:
TEACHER = "Qwen/Qwen3.5-9B"
NUM_CODING_TASKS = 800     # coding tasks written from docs passages that contain GDScript
NUM_DOCS_QUESTIONS = 200   # API questions taken from the docs dataset
NUM_EVAL_TASKS = 20        # held-out coding tasks for evaluating the 2B later (never solved here)
BATCH_SIZE = 64            # conversations generated together; vLLM batches them on the GPU
MAX_ROUNDS = 6             # model replies per conversation (each tool call costs one)
MAX_REPLY_TOKENS = 8192    # per reply; a reply that hits it never finished thinking and is dropped
MAX_MODEL_LEN = 32768      # longest conversation the teacher can handle
MAX_EXAMPLE_TOKENS = 8192  # longest example kept; the 2B is trained with this length
OUTPUT_DIR = "reasoning_examples"  # use a Google Drive folder to keep progress if Colab disconnects
HF_DATASET = "YOUR_HF_USERNAME/godot-reasoning-examples"

## 3. Tools

The Godot checker runs each check in its own temporary project, so several checks can run at once.

In [ ]:
import hashlib, os, pathlib, subprocess, urllib.request, zipfile

GODOT_VERSION = "4.7.2-stable"
GODOT_DIR = pathlib.Path("godot").resolve()
GODOT = str(GODOT_DIR / f"Godot_v{GODOT_VERSION}_linux.x86_64")

if not os.path.exists(GODOT):
    release = f"https://github.com/godotengine/godot/releases/download/{GODOT_VERSION}"
    archive = f"Godot_v{GODOT_VERSION}_linux.x86_64.zip"
    zip_path, _ = urllib.request.urlretrieve(f"{release}/{archive}")
    sums = urllib.request.urlopen(f"{release}/SHA512-SUMS.txt").read().decode()
    expected = next(line.split()[0] for line in sums.splitlines() if line.endswith(archive))
    actual = hashlib.sha512(pathlib.Path(zip_path).read_bytes()).hexdigest()
    assert actual == expected, "Godot download doesn't match the official SHA-512 checksum"
    zipfile.ZipFile(zip_path).extractall(GODOT_DIR)
    os.chmod(GODOT, 0o755)
print(subprocess.run([GODOT, "--version"], capture_output = True, text = True).stdout)

In [ ]:
import pathlib, re, subprocess, tempfile

SCRIPT_ERROR = re.compile(r"SCRIPT ERROR: (?:Parse Error: )?(.*)\n\s*at: .*?:(\d+)\)")

def check_gdscript(code: str) -> str:
    """Checks a GDScript file with Godot's own parser (no code is run) and returns the errors."""
    with tempfile.TemporaryDirectory() as project:  # a separate project per check, so checks can run in parallel
        (pathlib.Path(project) / "project.godot").write_text("config_version=5\n")
        (pathlib.Path(project) / "check.gd").write_text(code, encoding = "utf-8")
        try:
            result = subprocess.run(
                [GODOT, "--headless", "--path", project, "--check-only", "--script", "res://check.gd"],
                stdout = subprocess.PIPE, stderr = subprocess.STDOUT, text = True, timeout = 60)
        except subprocess.TimeoutExpired:
            return "The check timed out."
    errors = [f"Line {line}: {message}" for message, line in SCRIPT_ERROR.findall(result.stdout)]
    if errors:
        return "Errors:\n" + "\n".join(errors)
    if result.returncode != 0:  # failed without the usual error format: show Godot's output
        return "Errors:\n" + result.stdout[-1500:]
    return "No errors found."

print(check_gdscript("extends Node2D\n\nfunc _ready() -> void:\n    velocity = Vector2.ZERO\n"))

In [ ]:
from datasets import load_dataset, Features, Value

columns = ["instruction", "input", "output", "category", "source"]
docs = load_dataset("arunsrajan/godot-docs-dataset", split = "train", keep_default_na = False,
                    features = Features({column: Value("string") for column in columns})).to_list()
print(len(docs), "rows")

In [ ]:
import math, re
from collections import Counter, defaultdict

STOPWORDS = {"the", "a", "an", "of", "in", "on", "to", "and", "or", "is", "are", "do", "does",
             "how", "what", "when", "why", "which", "i", "my", "me", "it", "for", "with", "godot",
             "explain", "about", "tell", "this", "that", "can", "you"}

def tokenize(text):
    words = [w for w in re.findall(r"[a-z0-9_]+", text.lower()) if w not in STOPWORDS]
    # "look_at" is also indexed as "look" and "at", so "look at" finds it.
    return words + [part for w in words if "_" in w for part in w.split("_") if part and part not in STOPWORDS]

class BM25:
    """Small keyword search index (BM25 ranking), no extra packages needed."""
    def __init__(self, texts, k1 = 1.5, b = 0.75):
        self.k1, self.b = k1, b
        self.postings = defaultdict(list)  # term -> [(doc_id, term_count)]
        self.lengths = []
        for doc_id, text in enumerate(texts):
            counts = Counter(tokenize(text))
            self.lengths.append(sum(counts.values()))
            for term, count in counts.items():
                self.postings[term].append((doc_id, count))
        self.avg_length = sum(self.lengths) / len(self.lengths)
        n = len(texts)
        self.idf = {term: math.log(1 + (n - len(p) + 0.5) / (len(p) + 0.5)) for term, p in self.postings.items()}

    def search(self, query, k = 3):
        scores = defaultdict(float)
        for term in set(tokenize(query)):
            for doc_id, count in self.postings.get(term, ()):
                norm = count + self.k1 * (1 - self.b + self.b * self.lengths[doc_id] / self.avg_length)
                scores[doc_id] += self.idf[term] * count * (self.k1 + 1) / norm
        return sorted(scores, key = scores.get, reverse = True)[:k]

In [ ]:
index = BM25([row["instruction"] + "\n" + row["output"] for row in docs])

In [ ]:
MAX_RESULT_CHARS = 1500
DOCS_URL = "https://docs.godotengine.org/en/latest/"

def format_results(doc_ids):
    parts = []
    for rank, doc_id in enumerate(doc_ids, 1):
        row = docs[doc_id]
        text = row["output"]
        if len(text) > MAX_RESULT_CHARS:
            text = text[:MAX_RESULT_CHARS] + " ..."
        parts.append(f"[{rank}] {DOCS_URL}{row['source']}\n{text}")
    return "\n\n".join(parts) if parts else "No results."

def search_godot_docs(query: str) -> str:
    return format_results(index.search(query, k = 3))

TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_godot_docs",
        "description": "Search the official Godot Engine 4.7 documentation (class reference, tutorials and manuals). "
                       "Returns the 3 most relevant passages with their URLs.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string",
                          "description": "What to look up, e.g. 'Node2D look_at' or 'detect collisions CharacterBody2D'."},
            },
            "required": ["query"],
        },
    },
}]

In [ ]:
TOOLS.append({
    "type": "function",
    "function": {
        "name": "check_gdscript",
        "description": "Checks a complete GDScript file with the Godot 4 engine's own parser and returns the parse and "
                       "type errors with line numbers, or 'No errors found.'. It does not run the code, so it cannot "
                       "find logic errors. It cannot check C#.",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {"type": "string",
                         "description": "The complete GDScript file, starting with its extends line."},
            },
            "required": ["code"],
        },
    },
})
TOOL_FUNCTIONS = {"search_godot_docs": search_godot_docs, "check_gdscript": check_gdscript}

The system prompt is the same as in the [coding assistant notebook](https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_Godot_Coding_Assistant.ipynb).

In [ ]:
SYSTEM_PROMPT = (
    "You are a Godot Engine 4 coding assistant for GDScript and C#.\n"
    "- Use search_godot_docs to look up any Godot class, method, property or signal you are not sure about. "
    "Use only APIs that exist in Godot 4; never invent them.\n"
    "- Write GDScript with static types, for example `func _physics_process(delta: float) -> void:`, "
    "so the checker can find type errors.\n"
    "- Before answering with GDScript, call check_gdscript on the complete script and fix every error it reports. "
    "It only finds parse and type errors, so also check the logic yourself.\n"
    "- check_gdscript cannot check C#. For C#, follow the C# examples in the search results.\n"
    "- Give the final code in one fenced code block, then explain it briefly."
)

In [ ]:
TOOL_CALL = re.compile(r"<tool_call>\s*<function=([\w.-]+)>(.*?)</function>\s*</tool_call>", re.DOTALL)
PARAMETER = re.compile(r"<parameter=([\w.-]+)>\n?(.*?)\n?</parameter>", re.DOTALL)

def parse_tool_calls(reply):
    """Reads Qwen3.5's '<tool_call><function=name><parameter=x>...' blocks from a model reply."""
    calls = [{"name": name, "arguments": {k: v.strip() for k, v in PARAMETER.findall(body)}}
             for name, body in TOOL_CALL.findall(reply)]
    return calls, TOOL_CALL.sub("", reply).strip()


GDSCRIPT_BLOCK = re.compile(r"```(?:gdscript|gd)\n(.*?)```", re.DOTALL)

def run_tool(call):
    function = TOOL_FUNCTIONS.get(call["name"])
    if function is None:
        return f"Unknown tool: {call['name']}"
    try:
        return function(**call["arguments"])
    except TypeError as error:
        return f"Wrong arguments for {call['name']}: {error}"

## 4. Load Qwen3.5-9B with vLLM

`language_model_only=True` skips Qwen3.5's vision encoder. Loading takes a few minutes.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

llm = LLM(
    model = TEACHER,
    max_model_len = MAX_MODEL_LEN,
    gpu_memory_utilization = 0.90,
    language_model_only = True,  # text only: skips Qwen3.5's vision encoder
    seed = 3407,
    # quantization = "fp8",      # on an L4 (24 GB): halves the weights so long conversations fit
)
tokenizer = AutoTokenizer.from_pretrained(TEACHER)

Qwen3.5's recommended sampling for thinking-mode coding. Prompts are built with the model's own chat template, in
thinking mode, with both tools.

In [ ]:
THINKING_SAMPLING = SamplingParams(temperature = 0.6, top_p = 0.95, top_k = 20, max_tokens = MAX_REPLY_TOKENS)
TASK_SAMPLING = SamplingParams(temperature = 0.7, top_p = 0.8, top_k = 20, max_tokens = 200)

def count_tokens(text):
    return len(tokenizer(text, add_special_tokens = False)["input_ids"])

def render(messages):
    """The prompt for the teacher's next reply, in thinking mode with both tools."""
    return tokenizer.apply_chat_template(messages, tools = TOOLS, add_generation_prompt = True,
                                         tokenize = False, enable_thinking = True)

## 5. Write the tasks

The task list is saved to `OUTPUT_DIR/tasks.json`, so a rerun reuses it instead of writing new tasks.

In [ ]:
import json, os, random

TASK_PROMPT = """Below is a passage from the Godot 4 documentation. Write one realistic request that a Godot developer could ask a coding assistant, which this passage helps answer.
- Ask for working GDScript code for a small, concrete task: a script, a function, or a node's behavior.
- The request must make sense on its own. Don't mention the passage or the documentation.
- Reply with only the request, in one or two sentences.

Passage:
{passage}"""

def clean_task(text):
    text = text.strip().strip('"').strip()
    return text if 20 <= len(text) <= 500 and "\n\n" not in text else None

os.makedirs(OUTPUT_DIR, exist_ok = True)
tasks_path = os.path.join(OUTPUT_DIR, "tasks.json")
if os.path.exists(tasks_path):  # reuse the task list from an earlier run, so progress can resume
    tasks = json.load(open(tasks_path, encoding = "utf-8"))
else:
    rng = random.Random(3407)
    code_rows = [i for i, row in enumerate(docs) if "```gdscript" in row["output"]]
    rng.shuffle(code_rows)
    code_rows = code_rows[:int((NUM_CODING_TASKS + NUM_EVAL_TASKS) * 1.2)]  # extra, since some requests get dropped
    prompts = [tokenizer.apply_chat_template(
                   [{"role": "user", "content": TASK_PROMPT.format(passage = docs[i]["output"][:3000])}],
                   add_generation_prompt = True, tokenize = False, enable_thinking = False)
               for i in code_rows]
    outputs = llm.generate(prompts, TASK_SAMPLING)
    coding, seen = [], set()
    for i, output in zip(code_rows, outputs):
        task = clean_task(output.outputs[0].text)
        if task and task not in seen:
            seen.add(task)
            coding.append({"kind": "coding", "task": task, "source": docs[i]["source"]})
    coding = coding[:NUM_CODING_TASKS + NUM_EVAL_TASKS]

    questions = [i for i, row in enumerate(docs) if "(part " not in row["instruction"] and len(row["output"]) <= 1500]
    questions = rng.sample(questions, NUM_DOCS_QUESTIONS)
    api = [{"kind": "docs", "task": docs[i]["instruction"], "source": docs[i]["source"]} for i in questions]

    tasks = [{"id": n, "split": "eval" if n < NUM_EVAL_TASKS else "train", **task}
             for n, task in enumerate(coding + api)]
    json.dump(tasks, open(tasks_path, "w", encoding = "utf-8"), ensure_ascii = False, indent = 1)

eval_tasks = [t for t in tasks if t["split"] == "eval"]
train_tasks = [t for t in tasks if t["split"] == "train"]
print(f"{len(train_tasks)} tasks to solve, {len(eval_tasks)} held out for evaluation")
for task in train_tasks[:5]:
    print("-", task["task"])

## 6. Solve and check

`solve()` runs the tool loop for a whole batch at once. `verify()` decides which conversations become training examples.

In [ ]:
import concurrent.futures

def solve(batch):
    """Runs the tool loop for a batch of tasks at once: every round, all unfinished conversations get one reply."""
    conversations = [{**task, "answer": None, "failure": None, "tools_used": [],
                      "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                                   {"role": "user", "content": task["task"]}]} for task in batch]
    active = conversations
    for _ in range(MAX_ROUNDS):
        ready, prompts = [], []
        for conversation in active:
            prompt = render(conversation["messages"])
            if count_tokens(prompt) + MAX_REPLY_TOKENS > MAX_MODEL_LEN:
                conversation["failure"] = "conversation too long"
            else:
                ready.append(conversation)
                prompts.append(prompt)
        if not ready:
            break
        outputs = llm.generate(prompts, THINKING_SAMPLING, use_tqdm = False)
        active, calls_to_run = [], []
        for conversation, output in zip(ready, outputs):
            reply = output.outputs[0]
            if reply.finish_reason != "stop" or "</think>" not in reply.text:
                conversation["failure"] = "never finished thinking"
                continue
            reasoning, rest = reply.text.split("</think>", 1)
            calls, content = parse_tool_calls(rest.strip())
            message = {"role": "assistant", "content": content, "reasoning_content": reasoning.replace("<think>", "").strip()}
            if calls:
                message["tool_calls"] = [{"type": "function", "function": call} for call in calls]
                calls_to_run += [(conversation, call) for call in calls]
                active.append(conversation)
            else:
                conversation["answer"] = content
            conversation["messages"].append(message)
        # Run the tool calls in parallel: each Godot check takes about a second.
        with concurrent.futures.ThreadPoolExecutor(8) as pool:
            results = list(pool.map(lambda item: run_tool(item[1]), calls_to_run))
        for (conversation, call), result in zip(calls_to_run, results):
            conversation["messages"].append({"role": "tool", "content": result})
            conversation["tools_used"].append(call["name"])
    for conversation in active:
        conversation["failure"] = conversation["failure"] or "too many tool calls"
    return conversations

In [ ]:
def verify(conversation):
    """Returns why a conversation is rejected, or None if it is a good training example."""
    if conversation["failure"]:
        return conversation["failure"]
    answer = conversation["answer"] or ""
    if not answer.strip():
        return "empty answer"
    if "search_godot_docs" not in conversation["tools_used"]:
        return "didn't search the docs"
    blocks = GDSCRIPT_BLOCK.findall(answer)
    if conversation["kind"] == "coding":
        if not blocks:
            return "no GDScript in the answer"
        if "check_gdscript" not in conversation["tools_used"]:
            return "didn't check its code"
    for code in blocks:
        if check_gdscript(code) != "No errors found.":
            return "final GDScript has errors"
    text = tokenizer.apply_chat_template(conversation["messages"], tools = TOOLS, tokenize = False)
    if count_tokens(text) > MAX_EXAMPLE_TOKENS:
        return "longer than MAX_EXAMPLE_TOKENS"
    return None

Every conversation is saved to `OUTPUT_DIR/results.jsonl`, accepted or not, with the reason.

In [ ]:
import collections, time

results_path = os.path.join(OUTPUT_DIR, "results.jsonl")
done = set()
if os.path.exists(results_path):
    with open(results_path, encoding = "utf-8") as f:
        done = {json.loads(line)["id"] for line in f}
todo = [task for task in train_tasks if task["id"] not in done]
print(f"{len(done)} tasks already done, {len(todo)} to go")

for start in range(0, len(todo), BATCH_SIZE):
    started = time.time()
    batch = solve(todo[start:start + BATCH_SIZE])
    reasons = collections.Counter()
    with open(results_path, "a", encoding = "utf-8") as f:
        for conversation in batch:
            reason = verify(conversation)
            reasons[reason or "accepted"] += 1
            f.write(json.dumps({"id": conversation["id"], "kind": conversation["kind"], "task": conversation["task"],
                                "source": conversation["source"], "accepted": reason is None, "reason": reason or "",
                                "messages": conversation["messages"]}, ensure_ascii = False) + "\n")
    print(f"Batch {start // BATCH_SIZE + 1}/{-(-len(todo) // BATCH_SIZE)}: {reasons['accepted']}/{len(batch)} accepted "
          f"in {time.time() - started:.0f}s | {dict(reasons)}")

## 7. Look at the results

In [ ]:
with open(results_path, encoding = "utf-8") as f:
    results = [json.loads(line) for line in f]
accepted = [r for r in results if r["accepted"]]
print(f"{len(accepted)} of {len(results)} accepted")
print(collections.Counter(r["reason"] or "accepted" for r in results).most_common())
print(collections.Counter(r["kind"] for r in accepted))

In [ ]:
example = next(r for r in accepted if r["kind"] == "coding")
print("TASK:", example["task"])
for message in example["messages"][2:]:
    if message["role"] == "assistant":
        print("\n[thinking]", message["reasoning_content"][:600], "..." if len(message["reasoning_content"]) > 600 else "")
        for call in message.get("tool_calls", []):
            print("[tool call]", call["function"]["name"], str(call["function"]["arguments"])[:300])
        if message["content"]:
            print("[answer]", message["content"][:1500])
    else:
        print("[tool result]", message["content"][:300])

## 8. Push to Hugging Face

Set `HF_DATASET` in section 2 first. The dataset is private. `train` has the accepted conversations, with
`messages` as JSON; `test` has the held-out tasks, without answers.

In [ ]:
from datasets import Dataset, DatasetDict
from huggingface_hub import login

login()  # paste a Hugging Face token with write access
columns = lambda r, messages: {"id": r["id"], "kind": r["kind"], "task": r["task"], "source": r["source"], "messages": messages}
DatasetDict({
    "train": Dataset.from_list([columns(r, json.dumps(r["messages"], ensure_ascii = False)) for r in accepted]),
    "test": Dataset.from_list([columns(t, "") for t in eval_tasks]),  # held-out tasks, no answers
}).push_to_hub(HF_DATASET, private = True)
print(f"https://huggingface.co/datasets/{HF_DATASET}")